In [ ]:
import sys, os
sys.path.append(os.path.join(os.getcwd(), '../../')) # Add root of repo to import MBM

import warnings
import massbalancemachine as mbm
import logging
from datetime import datetime
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import torch 
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from tqdm.auto import tqdm
import json


from regions.TF_Europe.scripts.config_TF_Europe import *
from regions.TF_Europe.scripts.dataset import *
from regions.TF_Europe.scripts.plotting import *
from regions.TF_Europe.scripts.models import *
from regions.TF_Europe.scripts.utils import *

warnings.filterwarnings('ignore')
%load_ext autoreload
%autoreload 2

cfg = mbm.EuropeTFConfig()
mbm.utils.seed_all(cfg.seed)
mbm.utils.free_up_cuda()
mbm.plots.use_mbm_style()

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Initialize logging
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(message)s')


## Norway to Iceland:

In [ ]:
EXPERIMENT_NUMBER = 4
BASE_RESULTS_DIR = Path("results")
RESULTS_DIR = BASE_RESULTS_DIR / f"NOR_ISL_experiment_{EXPERIMENT_NUMBER}"

In [ ]:
N_SEEDS = 10
SEEDS = range(1, 1 + N_SEEDS)

df_results_adapter = pd.read_csv(os.path.join(RESULTS_DIR,
                                              "combined_results_adapter.csv"),
                                 index_col="exp_key")
df_results_dan = pd.read_csv(os.path.join(RESULTS_DIR,
                                          "combined_results_dan.csv"),
                             index_col="exp_key")

# filter dfs to only include the seeds we want
df_results_adapter = df_results_adapter[df_results_adapter["seed"].isin(SEEDS)]
df_results_dan = df_results_dan[df_results_dan["seed"].isin(SEEDS)]

In [ ]:
def plot_sweep_Y_by_G_colormap_ax(
    df,
    E_ZERO_a,
    E_FULL_a,
    E_ZERO_w,
    E_FULL_w,
    E_ZERO,
    E_FULL,
    metric="annual",
    band=(0.25, 0.75),
    title="Monitoring performance vs years monitored",
    ax=None,
    show_legend=True,
    show_colorbar=True,
):
    """Same as plot_sweep_Y_by_G_colormap, but draws into `ax` (no new figure)."""
    if ax is None:
        ax = plt.gca()

    if metric == "annual":
        rmse_col = "RMSE_annual"
        E0, EF = E_ZERO_a, E_FULL_a
        ylab = "RMSE_annual"
    elif metric == "winter":
        rmse_col = "RMSE_winter"
        E0, EF = E_ZERO_w, E_FULL_w
        ylab = "RMSE_winter"
    elif metric == "mean":
        rmse_col = "RMSE_TL"
        E0, EF = E_ZERO, E_FULL
        ylab = "RMSE (mean annual+winter)"
    else:
        raise ValueError(f"Unknown metric: {metric}")

    qlo, qhi = band

    g = (df.groupby(["G", "Y"])[rmse_col].agg(
        med="median",
        qlo=lambda s: np.quantile(s, qlo),
        qhi=lambda s: np.quantile(s, qhi),
        n=("size"),
    ).reset_index())

    G_vals = np.array(sorted(g["G"].unique()))
    norm = mpl.colors.Normalize(vmin=G_vals.min(), vmax=G_vals.max())
    cmap = mpl.cm.viridis_r

    for G_val in G_vals:
        sub = g[g["G"] == G_val].sort_values("Y")
        color = cmap(norm(G_val))

        ax.plot(
            sub["Y"],
            sub["med"],
            marker="o",
            linewidth=2,
            color=color,
            label=f"G={G_val}",
        )
        ax.fill_between(
            sub["Y"],
            sub["qlo"],
            sub["qhi"],
            alpha=0.15,
            color=color,
        )

    # baselines
    ax.axhline(E0,
               linestyle="--",
               color="black",
               linewidth=1,
               label="E_ZERO (NOR only)")
    ax.axhline(EF,
               linestyle=":",
               color="black",
               linewidth=1.5,
               label="E_FULL (max monitoring)")

    ax.set_xlabel("Y (years monitored)")
    ax.set_ylabel(f"{ylab} (median + {int(qlo*100)}–{int(qhi*100)}% band)")
    ax.set_title(title)

    if show_legend:
        ax.legend(loc="upper right", ncol=2, fontsize=9, frameon=False)

    if show_colorbar:
        sm = mpl.cm.ScalarMappable(norm=norm, cmap=cmap)
        sm.set_array([])
        cbar = plt.colorbar(sm, ax=ax, pad=0.02)
        cbar.set_label("G (number of glaciers)")

    return ax


def _get_baselines_from_df(df):
    """Your dfs have these as columns; take the (unique) value."""

    def one(col):
        vals = df[col].dropna().unique()
        if len(vals) == 0:
            raise ValueError(f"Column {col} has no non-NaN values.")
        if len(vals) > 1:
            # still OK; take median to be robust
            return float(np.median(vals))
        return float(vals[0])

    return dict(
        E_ZERO_a=one("E_ZERO_a"),
        E_FULL_a=one("E_FULL_a"),
        E_ZERO_w=one("E_ZERO_w"),
        E_FULL_w=one("E_FULL_w"),
        E_ZERO=one("E_ZERO"),
        E_FULL=one("E_FULL"),
    )


def plot_adapter_vs_dan_2x2(
        df_results_adapter,
        df_results_dan,
        band=(0.25, 0.75),
        suptitle=None,
        figsize=(14, 9),
        sharey=True,
):
    b_ad = _get_baselines_from_df(df_results_adapter)
    b_dn = _get_baselines_from_df(df_results_dan)

    fig, axes = plt.subplots(2, 2, figsize=figsize, sharey="row", sharex=True)
    # force y tick labels on all panels
    for ax in axes.flat:
        ax.tick_params(labelleft=True)
    # For cleanliness: one legend + one colorbar only
    # (put them in top-left panel)
    plot_sweep_Y_by_G_colormap_ax(
        df_results_adapter,
        **b_ad,
        metric="annual",
        band=band,
        title="Adapter LSTM — annual",
        ax=axes[0, 0],
        show_legend=True,
        show_colorbar=False,
    )

    plot_sweep_Y_by_G_colormap_ax(
        df_results_dan,
        **b_dn,
        metric="annual",
        band=band,
        title="DAN LSTM — annual",
        ax=axes[0, 1],
        show_legend=False,
        show_colorbar=False,
    )

    plot_sweep_Y_by_G_colormap_ax(
        df_results_adapter,
        **b_ad,
        metric="winter",
        band=band,
        title="Adapter LSTM — winter",
        ax=axes[1, 0],
        show_legend=False,
        show_colorbar=False,
    )

    plot_sweep_Y_by_G_colormap_ax(
        df_results_dan,
        **b_dn,
        metric="winter",
        band=band,
        title="DAN LSTM — winter",
        ax=axes[1, 1],
        show_legend=False,
        show_colorbar=False,
    )

    if suptitle:
        fig.suptitle(suptitle, y=0.98)

    plt.tight_layout()
    plt.show()
    return fig, axes

In [ ]:
plot_adapter_vs_dan_2x2(
    df_results_adapter,
    df_results_dan,
    band=(0.25, 0.75),
    suptitle=f"Iceland monitoring performance (source: NOR) - K ({N_SEEDS})",
    sharey=True,
)